# Fine-Tuning Ablation Run: MMS_LORA
### Model: MMS | Approach: LORA

This notebook implements the ASR fine-tuning ablation run. It:
1. Mounts Google Drive and loads the central experiment configs.
2. Initializes an isolated, resume-safe MLflow run.
3. Loads the processed dataset train/val/test splits.
4. Evaluates the zero-shot baseline performance on the test split.
5. Configures adaptation (layer freezing or PEFT LoRA) and prints parameters.
6. Runs fine-tuning with automatic epoch checkpoint-resume.
7. Runs final test-set evaluation and exports standard diagnostics plots.

### Stage 1 & 2: Environment setup, dependencies, and loading config

In [1]:
# 1. Colab Google Drive Mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/VAD
except ImportError:
    print("Not running on Google Colab. Operating in local project folder.")

# 2. Install missing packages
try:
    import transformers
    import datasets
    import peft
    import mlflow
    import ctc_segmentation
    import jiwer
    import soundfile
    print("All key dependencies are already installed.")
except ImportError:
    print("Installing dependencies...")
    !pip uninstall -y -q torchao
    !pip install -q -r requirements.txt
    # Reinstall ctc-segmentation from source to ensure compatibility with active NumPy (which is typically NumPy 2.x in Colab)
    !pip install -q --no-binary ctc-segmentation ctc-segmentation

# 3. Add src/ to Python path
import sys
import os
sys.path.append('src')
print("System path updated. Working directory contents:", os.listdir('.'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1FpN2rkyCNP6qnzVDYzzst46X-surMQRW/VAD


All key dependencies (including NumPy 1.x) are already installed.
System path updated.


In [2]:
from config_loader import load_config

# Load config for this specific experiment
config = load_config("mms_lora")
print("Loaded Configuration:")
for k, v in config.to_dict().items():
    print(f"  {k}: {v}")

Loaded Configuration:
  project_root: /content/drive/MyDrive/VAD
  seed: 42
  sample_rate: 16000
  video_url: https://www.youtube.com/watch?v=LPre3ILXY1k
  segment_min_duration_sec: 3
  segment_target_duration_sec: [5, 10]
  segment_max_duration_sec: 12
  alignment_confidence_drop_percentile: 10
  cross_model_agreement_drop_percentile: 10
  qa_sample_size: 18
  qa_sample_strata: 3
  qa_problem_rate_retune_threshold: 0.25
  split_ratios: [0.8, 0.1, 0.1]
  model_id: facebook/mms-1b-all
  lora: {'r': 8, 'alpha': 16, 'dropout': 0.05, 'target_modules': ['q_proj', 'v_proj']}
  regularization: {'mask_time_prob': 0.05, 'mask_feature_prob': 0.05}
  experiment_name: mms_lora
  model_config: mms
  approach: lora
  output_dir: runs/mms_lora
  learning_rate: 0.0002


In [3]:
import mlflow
from mlflow_utils import setup_mlflow_run

# Setup resume-safe MLflow run tracking
run = setup_mlflow_run(
    experiment_name=config.experiment_name,
    project_root=config.project_root
)

Found active run ID. Resuming MLflow run: '2ba0c38d2c1045e7b74017f5ed25e100'


### Stage 3: Load Splits Manifests

In [4]:
import pandas as pd

# Load manifest splits
train_df = pd.read_csv("data/splits/train.csv")
val_df = pd.read_csv("data/splits/val.csv")
test_df = pd.read_csv("data/splits/test.csv")

print(f"Split sizes:")
print(f"  Train: {len(train_df)} segments ({train_df['duration'].sum() / 60.0:.2f} mins)")
print(f"  Val  : {len(val_df)} segments ({val_df['duration'].sum() / 60.0:.2f} mins)")
print(f"  Test : {len(test_df)} segments ({test_df['duration'].sum() / 60.0:.2f} mins)")

Split sizes:
  Train: 85 segments (10.79 mins)
  Val  : 11 segments (1.42 mins)
  Test : 11 segments (1.30 mins)


### Stage 4: Load Model & Processor

In [5]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

# Monkey patch Wav2Vec2ForCTC to prevent PEFT/LoRA saving crashes
Wav2Vec2ForCTC.get_input_embeddings = lambda self: None
Wav2Vec2ForCTC.set_input_embeddings = lambda self, value: None


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

model_id = config.model_id
# Programmatic target language selection for MMS
from alignment import select_azerbaijani_adapter
base_processor = AutoProcessor.from_pretrained(model_id)
target_lang = select_azerbaijani_adapter(base_processor)

processor = AutoProcessor.from_pretrained(model_id, target_lang=target_lang)
model = Wav2Vec2ForCTC.from_pretrained(
    model_id,
    target_lang=target_lang,
    ignore_mismatched_sizes=True
)
model.to(device)

Running on device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Found Azerbaijani adapter codes in vocabulary: ['azb', 'azg', 'azj-script_cyrillic', 'azj-script_latin', 'azz']
Programmatically selected Azerbaijani adapter: 'azj-script_latin'


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

### Stage 5: Zero-Shot Baseline Evaluation

In [6]:
import datasets
import soundfile as sf
from eval_utils import ASRMetricEvaluator
from transformers import Trainer, TrainingArguments

# Construct evaluation dataset from splits
def load_audio(path):
    audio, sr = sf.read(path)
    return {"array": audio, "sampling_rate": sr}

def prepare_eval_dataset(batch):
    audio_data = load_audio(batch["audio_path"])
    if False:
        batch["input_features"] = processor(audio_data["array"], sampling_rate=16000).input_features[0]
    else:
        batch["input_values"] = processor(audio_data["array"], sampling_rate=16000).input_values[0]
    batch["labels"] = processor.tokenizer(batch["normalized_text"]).input_ids
    return batch

test_dataset = datasets.Dataset.from_pandas(test_df).map(prepare_eval_dataset)
print("Prepared test evaluation dataset.")

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Prepared test evaluation dataset.


In [7]:
# Evaluate zero-shot performance
evaluator = ASRMetricEvaluator(tokenizer=processor.tokenizer, is_ctc=True)

# Setup a dummy data collator for evaluation
class EvalCollator:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, features):
        input_key = "input_features" if False else "input_values"
        input_features = [{input_key: feature[input_key]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

    per_device_eval_batch_size=config.training.per_device_eval_batch_size,
eval_trainer = Trainer(
    model=model,
    args=dummy_args,
    eval_dataset=test_dataset,
    data_collator=EvalCollator(processor),
    compute_metrics=evaluator
)

print("Evaluating zero-shot baseline...")
baseline_res = eval_trainer.evaluate()
baseline_wer = baseline_res.get("eval_wer", 0.0)
baseline_cer = baseline_res.get("eval_cer", 0.0)

print("=" * 80)
print(f"Baseline Zero-shot WER: {baseline_wer:.4f} | CER: {baseline_cer:.4f}")
print("Generalization Caveat: Any ASR improvement here reflects speaker and channel adaptation on a same-video split.")
print("=" * 80)

# Log to MLflow
mlflow.log_metric("baseline_wer", baseline_wer)
mlflow.log_metric("baseline_cer", baseline_cer)

Evaluating zero-shot baseline...


Training Loss,Validation Loss,Step,Wer,Cer
No log,0.848475,0,0.386076,0.135163


Baseline Zero-shot WER: 0.3861 | CER: 0.1352
Generalization Caveat: Any ASR improvement here reflects speaker and channel adaptation on a same-video split.


### Stage 6: Apply Adaptation Approach (Freezing or LoRA)

In [8]:
from train_utils import configure_adaptation

# LoRA Module Naming Verification Step
if "lora" == "lora":
    print("Listing named modules to verify 'q_proj' and 'v_proj' module paths:")
    for name, module in list(model.named_modules())[:30]:
        print(f"  {name}")

# Configure frozen layers or LoRA config
model, trainable_params, total_params = configure_adaptation(
    model=model,
    approach=config.approach,
    model_type="mms"
)

# Log parameter stats to MLflow
mlflow.log_param("trainable_params", trainable_params)
mlflow.log_param("total_params", total_params)
mlflow.log_param("percent_trainable", 100.0 * trainable_params / total_params)

Listing named modules to verify 'q_proj' and 'v_proj' module paths:
  
  wav2vec2
  wav2vec2.feature_extractor
  wav2vec2.feature_extractor.conv_layers
  wav2vec2.feature_extractor.conv_layers.0
  wav2vec2.feature_extractor.conv_layers.0.conv
  wav2vec2.feature_extractor.conv_layers.0.layer_norm
  wav2vec2.feature_extractor.conv_layers.0.activation
  wav2vec2.feature_extractor.conv_layers.1
  wav2vec2.feature_extractor.conv_layers.1.conv
  wav2vec2.feature_extractor.conv_layers.1.layer_norm
  wav2vec2.feature_extractor.conv_layers.1.activation
  wav2vec2.feature_extractor.conv_layers.2
  wav2vec2.feature_extractor.conv_layers.2.conv
  wav2vec2.feature_extractor.conv_layers.2.layer_norm
  wav2vec2.feature_extractor.conv_layers.2.activation
  wav2vec2.feature_extractor.conv_layers.3
  wav2vec2.feature_extractor.conv_layers.3.conv
  wav2vec2.feature_extractor.conv_layers.3.layer_norm
  wav2vec2.feature_extractor.conv_layers.3.activation
  wav2vec2.feature_extractor.conv_layers.4
  wav2vec

0.20337966295601329

### Stage 7 & 8: Fine-Tuning Setup and Training

In [9]:
# Preprocess train and validation splits
train_dataset = datasets.Dataset.from_pandas(train_df).map(prepare_eval_dataset)
val_dataset = datasets.Dataset.from_pandas(val_df).map(prepare_eval_dataset)

print(f"Dataset splits preprocessed. Train={len(train_dataset)}, Val={len(val_dataset)}")

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Dataset splits preprocessed. Train=85, Val=11


In [10]:
# Configure data collators and regularization
# MMS Convolutional Regularization (analog to SpecAugment)
model.config.mask_time_prob = config.regularization.mask_time_prob
model.config.mask_feature_prob = config.regularization.mask_feature_prob
print(f"Set MMS mask_time_prob and mask_feature_prob to {config.regularization.mask_time_prob}")

data_collator = EvalCollator(processor)
eval_collator = data_collator


Set MMS mask_time_prob and mask_feature_prob to 0.05


In [11]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback

# Setup training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(config.output_dir, "checkpoints"),
    remove_unused_columns=False,
    learning_rate=config.learning_rate,
    weight_decay=0.01,
    per_device_train_batch_size=config.training.per_device_train_batch_size,
    per_device_eval_batch_size=config.training.per_device_eval_batch_size,
    gradient_accumulation_steps=config.training.gradient_accumulation_steps,
    num_train_epochs=15,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    metric_for_best_model="loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=config.training.gradient_checkpointing,
    logging_steps=5,
    report_to="none"
)

# Custom MLflow callback to log loss and evaluation curves per epoch
class MLflowLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs:
            if "loss" in logs:
                mlflow.log_metric("train_loss_step", logs["loss"], step=state.global_step)

    def on_epoch_end(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs:
            epoch = int(round(state.epoch))
            if "loss" in logs:
                mlflow.log_metric("train_loss_epoch", logs["loss"], step=epoch)
            if "eval_loss" in logs:
                mlflow.log_metric("eval_loss", logs["eval_loss"], step=epoch)
            if "eval_wer" in logs:
                mlflow.log_metric("eval_wer", logs["eval_wer"], step=epoch)
            if "eval_cer" in logs:
                mlflow.log_metric("eval_cer", logs["eval_cer"], step=epoch)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=evaluator,
    callbacks=[MLflowLoggingCallback()]
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
import os
from train_utils import get_last_checkpoint

final_weights_dir = os.path.join(config.output_dir, "checkpoints/final")
run_summary_path = os.path.join(config.output_dir, "run_summary.md")
training_completed = os.path.exists(final_weights_dir) and os.path.exists(run_summary_path)

if training_completed:
    print("=" * 80)
    print(f"TRAINING ALREADY COMPLETED: Final weights found at {final_weights_dir}")
    print("Loading fine-tuned model weights from final checkpoint and skipping training loop.")
    print("=" * 80)
    if config.approach == "lora":
        from peft import PeftModel
        from transformers import Wav2Vec2ForCTC
        print("Loading fresh base model for LoRA weights...")
        base_model = Wav2Vec2ForCTC.from_pretrained(config.model_id)
        base_model.to(device)
        model = PeftModel.from_pretrained(base_model, final_weights_dir)
    else:
        model = model.from_pretrained(final_weights_dir)
    model.to(device)
    trainer.model = model
else:
    # Check for existing checkpoints on Drive to support automatic resume
    last_chk = get_last_checkpoint(os.path.join(config.output_dir, "checkpoints"))
    print("Starting training loop...")
    trainer.train(resume_from_checkpoint=last_chk)

Starting training loop...


Epoch,Training Loss,Validation Loss,Wer,Cer
1,3.933944,0.711682,0.368421,0.131633
2,3.154114,0.613568,0.368421,0.132653
3,2.763520,0.573430,0.388158,0.130612
4,2.463526,0.509979,0.375000,0.119388
5,2.093272,0.447346,0.355263,0.105102
6,2.313267,0.394762,0.348684,0.102041
7,2.006406,0.352982,0.342105,0.097959
8,1.812195,0.334246,0.335526,0.090816
9,1.785949,0.323383,0.335526,0.087755
10,1.787829,0.317217,0.328947,0.085714


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetunin

### Stage 9: Final Test Split Evaluation

In [13]:
print("Running final test-set evaluation using best checkpoint...")
test_res = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
final_wer = test_res.get("test_wer", 0.0)
final_cer = test_res.get("test_cer", 0.0)

print(f"Final Fine-tuned test metrics -> WER: {final_wer:.4f} | CER: {final_cer:.4f}")
mlflow.log_metric("final_test_wer", final_wer)
mlflow.log_metric("final_test_cer", final_cer)

Running final test-set evaluation using best checkpoint...


Training Loss,Validation Loss,Epoch,Wer,Cer
1.643048,0.342125,15,0.278481,0.087398


Final Fine-tuned test metrics -> WER: 0.2785 | CER: 0.0874


### Stage 10 & 11: Export Visualizations & Save Summary

In [14]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Extract training logs to plot
history = trainer.state.log_history
step_loss = [h["loss"] for h in history if "loss" in h]
step_idx = [h["step"] for h in history if "loss" in h]

epoch_val_loss = [h["eval_loss"] for h in history if "eval_loss" in h]
epoch_val_wer = [h["eval_wer"] for h in history if "eval_wer" in h]
epoch_val_cer = [h["eval_cer"] for h in history if "eval_cer" in h]
epoch_idx = list(range(1, len(epoch_val_loss) + 1))

fig_dir = os.path.join("reports/figures", config.experiment_name)
os.makedirs(fig_dir, exist_ok=True)

if len(step_loss) > 0 and len(epoch_val_loss) > 0:
    # Plot 1: Train Loss per Step
    plt.figure()
    plt.plot(step_idx, step_loss, label="Train Loss (Step)", color="#4a90e2")
    plt.title(f"{config.experiment_name} - Training Loss Curve")
    plt.xlabel("Global Step")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(fig_dir, "train_loss_step.png"))
    plt.close()

    # Plot 2: Validation Loss vs Epoch
    plt.figure()
    plt.plot(epoch_idx, epoch_val_loss, marker="o", label="Val Loss", color="red")
    plt.title(f"{config.experiment_name} - Validation Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(fig_dir, "val_loss_epoch.png"))
    plt.close()

    # Plot 3: Validation WER/CER per Epoch
    plt.figure()
    plt.plot(epoch_idx, epoch_val_wer, marker="x", label="Val WER", color="blue")
    plt.plot(epoch_idx, epoch_val_cer, marker=".", label="Val CER", color="cyan")
    plt.title(f"{config.experiment_name} - Validation WER/CER Curves")
    plt.xlabel("Epoch")
    plt.ylabel("Error Rate")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(fig_dir, "val_metrics_epoch.png"))
    plt.close()
else:
    print("Log history is empty (training was skipped). Skipping Plots 1-3 training curves.")

# Plot 4: Baseline vs Fine-tuned Bar Chart
plt.figure()
metrics = ["WER", "CER"]
baseline_vals = [baseline_wer, baseline_cer]
final_vals = [final_wer, final_cer]
x_axis = np.arange(len(metrics))

plt.bar(x_axis - 0.2, baseline_vals, width=0.4, label="Baseline (Zero-shot)", color="#f5a623")
plt.bar(x_axis + 0.2, final_vals, width=0.4, label="Fine-tuned (Best)", color="#417505")
plt.xticks(x_axis, metrics)
plt.title(f"{config.experiment_name} - Performance Improvement")
plt.ylabel("Error Rate")
plt.legend()
plt.savefig(os.path.join(fig_dir, "performance_comparison.png"))
plt.close()

# Log all generated figures as artifacts in MLflow
mlflow.log_artifacts(fig_dir, artifact_path="figures")
print("Visualizations exported and logged.")

Visualizations exported and logged.


In [15]:
if not training_completed:
    final_weights_dir = os.path.join(config.output_dir, "checkpoints/final")
    if hasattr(model, "save_pretrained"):
        model.save_pretrained(final_weights_dir)
    processor.save_pretrained(final_weights_dir)
    print(f"Weights saved to {final_weights_dir}")

summary_text = f"""# Run Summary: {config.experiment_name}
- **Approach**: {config.approach}
- **Baseline Metrics**: WER={baseline_wer:.4f}, CER={baseline_cer:.4f}
- **Fine-tuned Metrics**: WER={final_wer:.4f}, CER={final_cer:.4f}
- **Trainable Parameters**: {trainable_params:,} (Total: {total_params:,})

## Generalization Caveat
This run's test split is a 10% holdout from the same video used for training.
Improvement over baseline here primarily reflects rapid adaptation to this speaker's voice
and this recording's acoustic conditions, not a general improvement in Azerbaijani ASR.
"""

with open(os.path.join(config.output_dir, "run_summary.md"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
mlflow.end_run()

Weights saved to runs/mms_lora/checkpoints/final
# Run Summary: mms_lora
- **Approach**: lora
- **Baseline Metrics**: WER=0.3861, CER=0.1352
- **Fine-tuned Metrics**: WER=0.2785, CER=0.0874
- **Trainable Parameters**: 1,966,080 (Total: 966,704,326)

## Generalization Caveat
This run's test split is a 10% holdout from the same video used for training.
Improvement over baseline here primarily reflects rapid adaptation to this speaker's voice
and this recording's acoustic conditions, not a general improvement in Azerbaijani ASR.



/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
